In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import duckdb as dd
import polars as pl
from skimpy import skim

from src.config import *
from src.const import (CATEGORICAL_NOMINAL_VARS, CATEGORICAL_ORDINAL_VARS,
                       NUMERICAL_CONTINUOUS_VARS, NUMERICAL_DISCRETE_VARS,
                       PRIMARY_KEY, TIME_VARS)
from src.data.dqa import data_quality_assessment
from src.data.schema import CreditCardBalanceSchema

In [13]:
# Establish DuckDB connection
os.chdir(DATABASE_DIR)
con = dd.connect(HOME_CREDIT_DB)
con

# **Data Wrangling**

In [14]:
query = "SELECT * FROM credit_card_balance"
df = con.sql(query).pl()

## **Data Diagnosis**



In [4]:
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                 Data Types                                                                │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                         │
│ ┃ Dataframe         ┃ Values  ┃ ┃ Column Type ┃ Count ┃                                                         │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                         │
│ │ Number of rows    │ 3840312 │ │ float64     │ 15    │                                                         │
│ │ Number of columns │ 23      │ │ int64       │ 7     │                                                         │
│ └───────────────────┴─────────┘ │ string      │ 1     │                                                         │
│                                 └─────────────┴───────┘                                                         │
│                                                     number                                                      │
│ ┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column  ┃ NA     ┃ NA %     ┃ mean    ┃ sd      ┃ p0      ┃ p25     ┃ p50     ┃ p75     ┃ p100    ┃ hist   ┃  │
│ ┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━┩  │
│ │ SK_ID_P │      0 │        0 │ 1905000 │  536500 │ 1000000 │ 1434000 │ 1897000 │ 2369000 │ 2843000 │ ▇▇▇▇▇▇ │  │
│ │ REV     │        │          │         │         │         │         │         │         │         │        │  │
│ │ SK_ID_C │      0 │        0 │  278300 │  102700 │  100000 │  189500 │  278400 │  367600 │  456200 │ ▇▇▇▇▇▇ │  │
│ │ URR     │        │          │         │         │         │         │         │         │         │        │  │
│ │ MONTHS_ │      0 │        0 │  -34.52 │   26.67 │     -96 │     -55 │     -28 │     -11 │      -1 │ ▂▃▃▃▅▇ │  │
│ │ BALANCE │        │          │         │         │         │         │         │         │         │        │  │
│ │ AMT_BAL │      0 │        0 │   58300 │  106300 │ -420300 │       0 │       0 │   89050 │ 1506000 │   ▇▁   │  │
│ │ ANCE    │        │          │         │         │         │         │         │         │         │        │  │
│ │ AMT_CRE │      0 │        0 │  153800 │  165100 │       0 │   45000 │  112500 │  180000 │ 1350000 │  ▇▁▁   │  │
│ │ DIT_LIM │        │          │         │         │         │         │         │         │         │        │  │
│ │ IT_ACTU │        │          │         │         │         │         │         │         │         │        │  │
│ │ AL      │        │          │         │         │         │         │         │         │         │        │  │
│ │ AMT_DRA │ 749816 │ 19.52487 │    5961 │   28230 │   -6827 │       0 │       0 │       0 │ 2115000 │   ▇    │  │
│ │ WINGS_A │        │ 19374884 │         │         │         │         │         │         │         │        │  │
│ │ TM_CURR │        │       13 │         │         │         │         │         │         │         │        │  │
│ │ ENT     │        │          │         │         │         │         │         │         │         │        │  │
│ │ AMT_DRA │      0 │        0 │    7433 │   33850 │   -6212 │       0 │       0 │       0 │ 2287000 │   ▇    │  │
│ │ WINGS_C │        │          │         │         │         │         │         │         │         │        │  │
│ │ URRENT  │        │          │         │         │         │         │         │         │         │        │  │
│ │ AMT_DRA │ 749816 │ 19.52487 │   288.2 │    8202 │       0 │       0 │       0 │       0 │ 1530000 │   ▇    │  │
│ │ WINGS_O │        │ 19374884 │         │         │         │         │         │         │         │        │  │
│ │ THER_CU │        │       13 │         │         │   

In [16]:
df.glimpse()

Rows: 3840312
Columns: 23
$ SK_ID_PREV                 <i64> 2562384, 2582071, 1740877, 1389973, 1891521, 2646502, 1079071, 2095912, 2181852, 1235299
$ SK_ID_CURR                 <i64> 378907, 363914, 371185, 337855, 126868, 380010, 171320, 118650, 367360, 203885
$ MONTHS_BALANCE             <i64> -6, -1, -7, -4, -1, -7, -6, -7, -4, -5
$ AMT_BALANCE                <f64> 56.97, 63975.555, 31815.225, 236572.11, 453919.455, 82903.815, 353451.645, 47962.125, 291543.075, 201261.195
$ AMT_CREDIT_LIMIT_ACTUAL    <i64> 135000, 45000, 450000, 225000, 450000, 270000, 585000, 45000, 292500, 225000
$ AMT_DRAWINGS_ATM_CURRENT   <f64> 0.0, 2250.0, 0.0, 2250.0, 0.0, 0.0, 67500.0, 45000.0, 90000.0, 76500.0
$ AMT_DRAWINGS_CURRENT       <f64> 877.5, 2250.0, 0.0, 2250.0, 11547.0, 0.0, 67500.0, 45000.0, 289339.425, 111026.7
$ AMT_DRAWINGS_OTHER_CURRENT <f64> 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
$ AMT_DRAWINGS_POS_CURRENT   <f64> 877.5, 0.0, 0.0, 0.0, 11547.0, 0.0, 0.0, 0.0, 199339.425, 34526.7

In [6]:
df.shape

(3840312, 23)

In [7]:
df.schema

Schema([('SK_ID_PREV', Int64),
        ('SK_ID_CURR', Int64),
        ('MONTHS_BALANCE', Int64),
        ('AMT_BALANCE', Float64),
        ('AMT_CREDIT_LIMIT_ACTUAL', Int64),
        ('AMT_DRAWINGS_ATM_CURRENT', Float64),
        ('AMT_DRAWINGS_CURRENT', Float64),
        ('AMT_DRAWINGS_OTHER_CURRENT', Float64),
        ('AMT_DRAWINGS_POS_CURRENT', Float64),
        ('AMT_INST_MIN_REGULARITY', Float64),
        ('AMT_PAYMENT_CURRENT', Float64),
        ('AMT_PAYMENT_TOTAL_CURRENT', Float64),
        ('AMT_RECEIVABLE_PRINCIPAL', Float64),
        ('AMT_RECIVABLE', Float64),
        ('AMT_TOTAL_RECEIVABLE', Float64),
        ('CNT_DRAWINGS_ATM_CURRENT', Float64),
        ('CNT_DRAWINGS_CURRENT', Int64),
        ('CNT_DRAWINGS_OTHER_CURRENT', Float64),
        ('CNT_DRAWINGS_POS_CURRENT', Float64),
        ('CNT_INSTALMENT_MATURE_CUM', Float64),
        ('NAME_CONTRACT_STATUS', String),
        ('SK_DPD', Int64),
        ('SK_DPD_DEF', Int64)])

In [8]:
df.columns

['SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'AMT_BALANCE',
 'AMT_CREDIT_LIMIT_ACTUAL',
 'AMT_DRAWINGS_ATM_CURRENT',
 'AMT_DRAWINGS_CURRENT',
 'AMT_DRAWINGS_OTHER_CURRENT',
 'AMT_DRAWINGS_POS_CURRENT',
 'AMT_INST_MIN_REGULARITY',
 'AMT_PAYMENT_CURRENT',
 'AMT_PAYMENT_TOTAL_CURRENT',
 'AMT_RECEIVABLE_PRINCIPAL',
 'AMT_RECIVABLE',
 'AMT_TOTAL_RECEIVABLE',
 'CNT_DRAWINGS_ATM_CURRENT',
 'CNT_DRAWINGS_CURRENT',
 'CNT_DRAWINGS_OTHER_CURRENT',
 'CNT_DRAWINGS_POS_CURRENT',
 'CNT_INSTALMENT_MATURE_CUM',
 'NAME_CONTRACT_STATUS',
 'SK_DPD',
 'SK_DPD_DEF']

In [10]:
df.null_count()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,749816,0,749816,749816,305236,767988,0,0,0,0,749816,0,749816,749816,305236,0,0,0


In [11]:
df.describe()

statistic,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64
"""count""",3.840312e6,3.840312e6,3.840312e6,3.840312e6,3.840312e6,3.090496e6,3.840312e6,3.090496e6,3.090496e6,3.535076e6,3.072324e6,3.840312e6,3.840312e6,3.840312e6,3.840312e6,3.090496e6,3.840312e6,3.090496e6,3.090496e6,3.535076e6,"""3840312""",3.840312e6,3.840312e6
"""null_count""",0.0,0.0,0.0,0.0,0.0,749816.0,0.0,749816.0,749816.0,305236.0,767988.0,0.0,0.0,0.0,0.0,749816.0,0.0,749816.0,749816.0,305236.0,"""0""",0.0,0.0
"""mean""",1.9045e6,278324.207289,-34.521921,58300.155262,153807.9574,5961.324822,7433.388179,288.169582,2968.804848,3540.204129,10280.537702,7588.856739,55965.876905,58088.811177,58098.285489,0.309449,0.703144,0.004812,0.559479,20.825084,null,9.283667,0.331622
"""std""",536469.470563,102704.475133,26.667751,106307.031024,165145.699525,28225.688578,33846.077333,8201.989345,20796.887047,5600.154122,36078.084953,32005.987768,102533.616846,105965.369908,105971.801104,1.100401,3.190347,0.082639,3.240649,20.051494,null,97.5157,21.479231
"""min""",1.000018e6,100006.0,-96.0,-420250.185,0.0,-6827.31,-6211.62,0.0,0.0,0.0,0.0,0.0,-423305.82,-420250.185,-420250.185,0.0,0.0,0.0,0.0,0.0,"""Active""",0.0,0.0
"""25%""",1.434385e6,189517.0,-55.0,0.0,45000.0,0.0,0.0,0.0,0.0,0.0,152.37,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,null,0.0,0.0
"""50%""",1.897122e6,278396.0,-28.0,0.0,112500.0,0.0,0.0,0.0,0.0,0.0,2702.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,null,0.0,0.0
"""75%""",2.369324e6,367580.0,-11.0,89046.585,180000.0,0.0,0.0,0.0,0.0,6633.9,9000.0,6750.0,85359.15,88899.48,88914.51,0.0,0.0,0.0,0.0,32.0,null,0.0,0.0
"""max""",2.843496e6,456250.0,-1.0,1.5059e6,1.35e6,2.115e6,2.2871e6,1.529847e6,2.2393e6,202882.005,4.2892e6,4.2783e6,1.4723e6,1.4933e6,1.4933e6,51.0,165.0,12.0,165.0,120.0,"""Signed""",3260.0,3260.0


### **Numerical Variables**

In [19]:
## Discrete Variables
for col in NUMERICAL_DISCRETE_VARS:
    print(col)
    print(df[col].n_unique())
    print(df[col].value_counts(normalize=True).sort(col))
    print("\n")

MONTHS_BALANCE
96
shape: (96, 2)
┌────────────────┬────────────┐
│ MONTHS_BALANCE ┆ proportion │
│ ---            ┆ ---        │
│ i64            ┆ f64        │
╞════════════════╪════════════╡
│ -96            ┆ 0.003052   │
│ -95            ┆ 0.00326    │
│ -94            ┆ 0.003489   │
│ -93            ┆ 0.003697   │
│ -92            ┆ 0.003883   │
│ …              ┆ …          │
│ -5             ┆ 0.026182   │
│ -4             ┆ 0.02659    │
│ -3             ┆ 0.026132   │
│ -2             ┆ 0.024645   │
│ -1             ┆ 0.016237   │
└────────────────┴────────────┘


CNT_DRAWINGS_ATM_CURRENT
45
shape: (45, 2)
┌──────────────────────────┬────────────┐
│ CNT_DRAWINGS_ATM_CURRENT ┆ proportion │
│ ---                      ┆ ---        │
│ f64                      ┆ f64        │
╞══════════════════════════╪════════════╡
│ null                     ┆ 0.195249   │
│ 0.0                      ┆ 0.694141   │
│ 1.0                      ┆ 0.055584   │
│ 2.0                      ┆ 0.024906   │


In [20]:
## Continuous Variables
for col in NUMERICAL_CONTINUOUS_VARS:
    print(col)
    print(df[col].n_unique())
    print(df[col].value_counts(normalize=True).sort(col))
    print("\n")

AMT_BALANCE
1347904
shape: (1_347_904, 2)
┌─────────────┬────────────┐
│ AMT_BALANCE ┆ proportion │
│ ---         ┆ ---        │
│ f64         ┆ f64        │
╞═════════════╪════════════╡
│ -420250.185 ┆ 2.6040e-7  │
│ -261471.015 ┆ 2.6040e-7  │
│ -259848.945 ┆ 2.6040e-7  │
│ -240305.985 ┆ 2.6040e-7  │
│ -223224.21  ┆ 5.2079e-7  │
│ …           ┆ …          │
│ 1.3293e6    ┆ 2.6040e-7  │
│ 1347979.5   ┆ 2.6040e-7  │
│ 1.3546e6    ┆ 2.6040e-7  │
│ 1.3548e6    ┆ 2.6040e-7  │
│ 1.5059e6    ┆ 2.6040e-7  │
└─────────────┴────────────┘


AMT_CREDIT_LIMIT_ACTUAL
181
shape: (181, 2)
┌─────────────────────────┬────────────┐
│ AMT_CREDIT_LIMIT_ACTUAL ┆ proportion │
│ ---                     ┆ ---        │
│ i64                     ┆ f64        │
╞═════════════════════════╪════════════╡
│ 0                       ┆ 0.196292   │
│ 4500                    ┆ 0.000289   │
│ 9000                    ┆ 0.000577   │
│ 13500                   ┆ 0.000628   │
│ 18000                   ┆ 0.000191   │
│ …      

### **Categorical Variables**

In [21]:
## Nominal Variables
for col in CATEGORICAL_NOMINAL_VARS:
    print(col)
    print(df[col].n_unique())
    print(df[col].value_counts(normalize=True).sort(col))
    print("\n")

SK_ID_PREV
104307
shape: (104_307, 2)
┌────────────┬────────────┐
│ SK_ID_PREV ┆ proportion │
│ ---        ┆ ---        │
│ i64        ┆ f64        │
╞════════════╪════════════╡
│ 1000018    ┆ 0.000001   │
│ 1000030    ┆ 0.000002   │
│ 1000031    ┆ 0.000004   │
│ 1000035    ┆ 0.000001   │
│ 1000077    ┆ 0.000003   │
│ …          ┆ …          │
│ 2843476    ┆ 0.000025   │
│ 2843477    ┆ 0.000022   │
│ 2843478    ┆ 0.000023   │
│ 2843493    ┆ 0.000004   │
│ 2843496    ┆ 0.000004   │
└────────────┴────────────┘


SK_ID_CURR
103558
shape: (103_558, 2)
┌────────────┬────────────┐
│ SK_ID_CURR ┆ proportion │
│ ---        ┆ ---        │
│ i64        ┆ f64        │
╞════════════╪════════════╡
│ 100006     ┆ 0.000002   │
│ 100011     ┆ 0.000019   │
│ 100013     ┆ 0.000025   │
│ 100021     ┆ 0.000004   │
│ 100023     ┆ 0.000002   │
│ …          ┆ …          │
│ 456244     ┆ 0.000011   │
│ 456246     ┆ 0.000002   │
│ 456247     ┆ 0.000025   │
│ 456248     ┆ 0.000006   │
│ 456250     ┆ 0.000003   

In [23]:
# Ordinal Variables
for col in CATEGORICAL_ORDINAL_VARS:
    print(col)
    print(df[col].n_unique())
    print(df[col].value_counts(normalize=True).sort(col))
    print("\n")

### **Time Variables**

In [24]:
# Ordinal Variables
for col in TIME_VARS:
    print(col)
    print(df[col].n_unique())
    print(df[col].value_counts(normalize=True).sort(col))
    print("\n")

## **Removing Duplicates**

In [ ]:
df = df.unique(subset=PRIMARY_KEY, keep="first")

## **Missing Values**

### **Single Imputation**

In [15]:
# Only fill nulls where necessary to meet schema requirements
df = df.with_columns(
    [
        pl.col("AMT_DRAWINGS_ATM_CURRENT").fill_null(0),
        pl.col("AMT_DRAWINGS_OTHER_CURRENT").fill_null(0),
        pl.col("AMT_DRAWINGS_POS_CURRENT").fill_null(0),
        pl.col("AMT_INST_MIN_REGULARITY").fill_null(0),
        pl.col("AMT_PAYMENT_CURRENT").fill_null(0),
        pl.col("CNT_DRAWINGS_ATM_CURRENT").fill_null(0),
        pl.col("CNT_DRAWINGS_OTHER_CURRENT").fill_null(0),
        pl.col("CNT_DRAWINGS_POS_CURRENT").fill_null(0),
        pl.col("CNT_INSTALMENT_MATURE_CUM").fill_null(0),
    ]
)

### **Deletion**

In [16]:
df = df.filter(pl.col("NAME_CONTRACT_STATUS") != "Unknown")

## **Data Type Transformation**

In [17]:
for col in NUMERICAL_DISCRETE_VARS:
    df = df.with_columns(pl.col(col).cast(pl.Int64))

for col in NUMERICAL_CONTINUOUS_VARS:
    df = df.with_columns(pl.col(col).cast(pl.Float64))

for col in CATEGORICAL_NOMINAL_VARS:
    df = df.with_columns(pl.col(col).cast(pl.Categorical))

for col in TIME_VARS:
    df = df.with_columns(pl.col(col).str.to_datetime())

In [9]:
df.head()

SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,cat,i64,i64
2562384,378907,-6,56.97,135000.0,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.0,0.0,0.0,0,1,0,1,35,"""Active""",0,0
2582071,363914,-1,63975.555,45000.0,2250.0,2250.0,0.0,0.0,2250.0,2250.0,2250.0,60175.08,64875.555,64875.555,1,1,0,0,69,"""Active""",0,0
1740877,371185,-7,31815.225,450000.0,0.0,0.0,0.0,0.0,2250.0,2250.0,2250.0,26926.425,31460.085,31460.085,0,0,0,0,30,"""Active""",0,0
1389973,337855,-4,236572.11,225000.0,2250.0,2250.0,0.0,0.0,11795.76,11925.0,11925.0,224949.285,233048.97,233048.97,1,1,0,0,10,"""Active""",0,0
1891521,126868,-1,453919.455,450000.0,0.0,11547.0,0.0,11547.0,22924.89,27000.0,27000.0,443044.395,453919.455,453919.455,0,1,0,1,101,"""Active""",0,0


# **Cleaned Consolidated Data (CCD)**

Cleaned Consolidated Data (CCD) serves as a consolidated and reconciled repository of historical internal and external credit data, and acts as the foundational dataset for all regulatory model development and monitoring. This includes transactional, behavioural, and reference data at the account, customer, and product level. It has been subject to data quality validation and reconciliation processes against core IT systems. These include:

- Reconciliation to general ledger balances and servicing systems;
- Data quality reviews to ensure accuracy and internal consistency (e.g., checking for orphan records, missing values, illogical transitions).

In [ ]:
query = f"""
        CREATE OR REPLACE TABLE {CREDIT_CARD_BALANCE_CCD} AS
        SELECT * FROM df
    """

con.execute(query)

In [ ]:
# Perform DQA on credit card balance data
query = f"SELECT COUNT(*) as total_rows FROM {CREDIT_CARD_BALANCE_CCD}"
total_rows = con.execute(query).fetchone()[0]

success, validated_data = data_quality_assessment(
    CREDIT_CARD_BALANCE_CCD, CreditCardBalanceSchema, con, sample_size=total_rows
)

if success:
    print(
        "\n✅ Credit card balance (Cleaned) data quality assessment completed successfully"
    )
else:
    print("\n❌ Credit card balance (Cleaned) data quality assessment failed")

query = f"""DROP TABLE {CREDIT_CARD_BALANCE_CCD};"""
con.execute(query)


DATA QUALITY ASSESSMENT: CREDIT_CARD_BALANCE_CCD

🔍 1. COMPLETENESS ASSESSMENT
----------------------------------------
Total records in credit_card_balance_ccd: 3,840,312
Sample size for validation: 3,840,312 rows

Completeness by column (% non-null):
  ✅ SK_ID_PREV: 100.0%
  ✅ SK_ID_CURR: 100.0%
  ✅ MONTHS_BALANCE: 100.0%
  ✅ AMT_BALANCE: 100.0%
  ✅ AMT_CREDIT_LIMIT_ACTUAL: 100.0%
  ✅ AMT_DRAWINGS_ATM_CURRENT: 100.0%
  ✅ AMT_DRAWINGS_CURRENT: 100.0%
  ✅ AMT_DRAWINGS_OTHER_CURRENT: 100.0%
  ✅ AMT_DRAWINGS_POS_CURRENT: 100.0%
  ✅ AMT_INST_MIN_REGULARITY: 100.0%
  ✅ AMT_PAYMENT_CURRENT: 100.0%
  ✅ AMT_PAYMENT_TOTAL_CURRENT: 100.0%
  ✅ AMT_RECEIVABLE_PRINCIPAL: 100.0%
  ✅ AMT_RECIVABLE: 100.0%
  ✅ AMT_TOTAL_RECEIVABLE: 100.0%
  ✅ CNT_DRAWINGS_ATM_CURRENT: 100.0%
  ✅ CNT_DRAWINGS_CURRENT: 100.0%
  ✅ CNT_DRAWINGS_OTHER_CURRENT: 100.0%
  ✅ CNT_DRAWINGS_POS_CURRENT: 100.0%
  ✅ CNT_INSTALMENT_MATURE_CUM: 100.0%
  ✅ NAME_CONTRACT_STATUS: 100.0%
  ✅ SK_DPD: 100.0%
  ✅ SK_DPD_DEF: 100.0%

🔍 2. 

In [ ]:
os.chdir(INTERIM_DATA_DIR)
df.write_parquet(f"{CREDIT_CARD_BALANCE_CCD}")

In [21]:
con.close()